# A Deep dive into KEY concepts for Computer Vision

## Table of Contents

1. [Section 0: Install Dependencies](#sec0)
2. [Section 1: Introduction](#sec1)
3. [Section 2: Anchor Boxes & Region Proposals](#sec2)
4. [Section 3: The Head — Classification & Regression (and RoI Align)](#sec3)
5. [Section 4: NMS (Non-Maximum Suppression)](#sec4)
6. [Section 5: Evaluation Metrics (Accuracy, Precision, Recall, F1, mAP@50, mAP@50–95)](#sec5)
7. [Section 6: Loss Functions (Multi-task Learning)](#sec6)
8. [References](#references)

<a name='sec0'></a>

# Section 0:  Install dependencies

In [ ]:
# === Requirements ===
# Colab already ships torch, torchvision, matplotlib, seaborn, scikit-learn, Pillow and numpy.
# Only `torchinfo` is missing, so that is all we install.
#
# Two things NOT to do here:
#   1. `uv pip install ...` — outside a virtualenv uv exits with
#      "No virtual environment found", and uv is not preinstalled in Colab anyway.
#      `!` never raises, so the notebook would keep going and die later on an import.
#      If you really want uv:  !pip install -q uv && uv pip install --system -q <pkgs>
#   2. Reinstalling torch/torchvision from PyPI — Colab's builds are CUDA-matched and the
#      generic wheels silently break the GPU runtime.
!pip install -q torchinfo


In [ ]:

import random
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple, Union

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image, ImageDraw
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from torchinfo import summary

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Keep notebook runtime bounded on CPU.
FAST_MODE = True

device = torch.device('cpu')
print('device:', device)

fig_dir = Path('figures')
fig_dir.mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 120,
})

<a name='sec1'></a>

# Section 1 - Introduction

Modern computer vision systems are built on a combination of **powerful architectures** and **carefully designed training strategies**. Understanding these components is key to mastering tasks such as **object detection**, **image classification**, and **representation learning**.

This section provides a structured deep dive into the most important concepts, blending **theory** with **practical intuition**.

---

### 🧩 What You’ll Explore

#### 🔹 Anchors

We start with **anchors**, a core concept in object detection that allows models to translate dense feature maps into **candidate object regions** across different scales and aspect ratios — and then contrast them with the **anchor-free** heads that most current detectors use.

#### 🔹 Detection Head + RoI Align

Next, we analyze how models refine predictions using the **detection head**, and how **RoI Align** extracts precise spatial features for accurate **localization and classification**.

#### 🔹 NMS

We look at **Non-Maximum Suppression**, the classic way to collapse hundreds of overlapping candidate boxes into one detection per object — why it has to run *per class*, and why the newest detectors no longer need it at all.

#### 🔹 Evaluation metrics

We connect **accuracy**, **precision**, **recall** and **F1** to classification and detection, and unpack **mAP@50** vs **mAP@50–95**—the standard localisation-aware scores used on COCO-style benchmarks.

#### 🔹 Loss Functions

A model learns through its **loss function**. We break down the classification losses (cross-entropy, focal) and the whole box-regression lineage (Smooth L1 → IoU → GIoU → DIoU → CIoU), how they combine into a multi-task objective, and what each one fixes in its predecessor.


<a name='sec2'></a>

# Section 2 - Anchor Boxes & Region Proposals
---


## 2.1 - IoU ( Intersection over Union )
---

Anchor-based detectors use predefined bounding box hypotheses tiled over feature maps. Training assigns each anchor to a ground-truth box (positive/negative/ignore) using IoU ( Intersection over Union ).

**IoU**:
$$\mathrm{IoU}(A,B)=\frac{|A\cap B|}{|A\cup B|}$$

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_3_Key_Concepts_for_Deep_Learning_CV/iou.webp">

### Assignment rules (Faster R-CNN's RPN)

The rules are applied **in this order** — the first one is the part most summaries drop:

1. **Max-IoU rule (applied first).** For **every** ground-truth box, the anchor with the *highest* IoU against it is labelled **positive**, even if that IoU is below the threshold below. The paper keeps this rule *"for the reason that in some rare cases the second condition may find no positive sample"*. Without it, a small or oddly-shaped object that **no** anchor overlaps at 0.7 would never produce a single positive sample and could never be learned.
2. **Threshold rule.** Any remaining anchor with **IoU > 0.7** against *any* ground-truth box is also **positive**.
3. **Negative.** An anchor with **IoU < 0.3** against *all* ground-truth boxes.
4. **Ignore.** Everything in between (0.3 ≤ IoU ≤ 0.7) contributes **no loss** at all.

> ⚠️ **The numbers are detector-specific, not universal.** 0.7 / 0.3 is the **RPN** setting.
> RetinaNet uses **0.5 / 0.4**, and Faster R-CNN's *second* stage (the RoI head, which scores the
> proposals the RPN already produced) uses a single **0.5** threshold. Anchor-free detectors
> (§2.2.5) replace this IoU rule entirely.

## 2.2 Anchor Boxes
---

### *Anchor Boxes in Object Detection*
Anchor boxes are the core idea behind the **anchor-based** family of detectors — Faster R-CNN, SSD, RetinaNet and YOLOv2–v5. The problem they solve: how do you detect objects of wildly different shapes and sizes in a single forward pass?

> **Heads-up:** most detectors a student would reach for *today* are **anchor-free** — YOLOv8/YOLO11/YOLO26 use an anchor-free split head, FCOS regresses distances to the four box edges, and DETR/RT-DETR predict boxes directly from learned queries. Anchors are still the clearest mental model for how dense detection works, and they are what Faster R-CNN, SSD and RetinaNet actually do, so we build them first and then contrast with the anchor-free view in **§2.2.5**.


### 2.2.1 *The Problem*
A naive detector might predict one bounding box per grid cell. But what if there's a tall person and a wide car in the same region? You need to simultaneously predict multiple boxes with different aspect ratios. That's where anchor boxes come in.

### 2.2.2 *What is an Anchor Box?*
An anchor box (also called a prior or default box) is a predefined reference bounding box placed at every location in a feature map. Instead of predicting absolute coordinates from scratch, the model learns small offsets (Δx, Δy, Δw, Δh) relative to each anchor.

### 2.2.3 How It Works

1. Feature map

<p>
The image passes through a CNN backbone, producing a feature map — a grid of spatial activations. Each cell represents a region of the original image.

Here we show a 6×6 feature map. In practice, detectors often have multiple feature maps at different scales (FPN).
</p>

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_3_Key_Concepts_for_Deep_Learning_CV/anchor-step1.png" width=400>

2. Anchor placement

<p>
At every grid cell, we place a set of anchor boxes centered on that cell's position.

Each anchor has a predefined aspect ratio: square (1:1), wide (2:1), or tall (1:2).

With 3 anchors × 36 cells = 108 anchor proposals for this tiny grid. Real detectors use thousands.
</p>

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_3_Key_Concepts_for_Deep_Learning_CV/anchor-step2.png" width=400>

3. Scale + ratio
<p>
Anchors also vary by scale (size). Combining 3 scales × 3 ratios gives 9 anchors per cell.

This ensures coverage of objects ranging from small coins to large buses in the same image.
</p>

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_3_Key_Concepts_for_Deep_Learning_CV/anchor-step3.png" width=400>

4. Predict offsets

For each anchor, the network predicts 4 offsets:

```text
• Δx, Δy — shift the center
• Δw — scale the width
• Δh — scale the height
```

These are small adjustments from the anchor, not absolute coordinates. This makes learning much easier.


<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_3_Key_Concepts_for_Deep_Learning_CV/anchor-step4.png" width=400>







### 2.2.4 The Math: Encoding & Decoding
The model doesn't predict raw pixel coordinates. Instead it predicts encoded offsets to keep gradients well-behaved:

Where *a = anchor properties*, *g = ground truth*, *t = target offsets*, *b = decoded prediction*. The log on width/height prevents the network from predicting negative sizes.

- Encoding (ground truth → training targets):
```text
tₓ = (gₓ - aₓ) / aᵥ
tᵧ = (gᵧ - aᵧ) / aₕ
tᵥ = log(gᵥ / aᵥ)
tₕ = log(gₕ / aₕ)
```

- Decoding (model output → final box):
```text
bₓ = tₓ · aᵥ + aₓ      ← predicted center x
bᵧ = tᵧ · aₕ + aᵧ      ← predicted center y
bᵥ = aᵥ · exp(tᵥ)      ← predicted width
bₕ = aₕ · exp(tₕ)      ← predicted height
```



### 2.2.5 Anchor-free detectors

Everything above assumes a fixed set of reference boxes. **Anchor-free** detectors throw them away, and they are now the mainstream choice.

**What replaces the anchor?** The feature-map *location* itself. For each point on the feature map, FCOS-style heads regress **four distances to the edges of the box** — how far left, top, right and bottom the object's border is from this point:

```text
Anchor-based:   predict (tx, ty, tw, th)   relative to a predefined box
Anchor-free:    predict (l, t, r, b)       distances from this pixel to the 4 box edges

                x1 = x_point − l      y1 = y_point − t
                x2 = x_point + r      y2 = y_point + b
```

**What this buys you**

- **No anchor hyperparameters.** Scales, aspect ratios, the number of anchors per cell and the IoU assignment thresholds all disappear. Those were dataset-specific knobs that had to be re-tuned whenever object statistics changed.
- **Far fewer outputs.** One prediction per location instead of *k* per location, so the head is cheaper and the positive/negative imbalance is smaller.

**But something still has to decide which locations are positive.** Anchor-free detectors replace the IoU rule with a *label assignment* strategy:

| Assignment | Used by | Idea |
|---|---|---|
| Centre sampling | FCOS | locations near the object centre are positive |
| **Task-aligned assignment (TAL)** | YOLOv8, YOLO11, YOLO26 | score each location by `classification^α × IoU^β` and take the top-*k* — the assignment follows the *joint* quality of class and box, not geometry alone |
| **Hungarian (bipartite) matching** | DETR, Deformable DETR, DINO-DETR, RT-DETR | solve a one-to-one assignment between *N* predictions and the ground-truth set, so exactly one prediction is responsible for each object |

Hungarian matching is what lets DETR skip NMS entirely (see **Section 4**): if only one prediction is ever assigned to an object, there are no duplicates to suppress.


<a name='sec3'></a>

# Section 3 - The Head: Classification & Regression (and RoI Align)

Detection heads split into two jobs:
- **Classification head**: predicts whether/which class an anchor region corresponds to.
- **Regression head**: predicts box refinements as deltas relative to anchors.

We also compare **RoI Pooling vs RoI Align**: RoI Align samples continuously (bilinear interpolation) to reduce misalignment.

## 3.1 Example: ResNet50 backbone and FPN neck

- **Backbone:** We use `torchvision.models.resnet50` up through `layer1`–`layer4` (no global pool or FC). Outputs are named **C2**–**C5** with channel widths 256, 512, 1024, and 2048 — the usual convention for detectors built on ResNet.

- **FPN neck:** Lateral 1×1 convolutions project each level to a common width (256), then a top-down path upsamples and sums from deep to shallow. A 3×3 convolution on each merged map yields **P2**–**P5**, which feed the shared detection head.

### 3.1.1 The problem

In a two-stage detector like Faster R-CNN, the backbone outputs a feature map at a lower resolution than the input image.

A Region Proposal Network (RPN) then proposes bounding boxes
<b> but those boxes live in input-image coordinates, not feature-map coordinates.</b>

To classify and refine each proposal, you need to extract a fixed-size feature crop from the feature map for every proposal.

In [ ]:
class ResNet50Backbone(nn.Module):
    """ResNet50 feature extractor returning multi-scale maps C2..C5."""

    def __init__(self, pretrained: bool = False) -> None:
        super().__init__()
        w = torchvision.models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        m = torchvision.models.resnet50(weights=w)
        self.conv1 = m.conv1
        self.bn1 = m.bn1
        self.relu = m.relu
        self.maxpool = m.maxpool
        self.layer1 = m.layer1
        self.layer2 = m.layer2
        self.layer3 = m.layer3
        self.layer4 = m.layer4

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        c2 = self.layer1(x)
        c3 = self.layer2(c2)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)
        return {'C2': c2, 'C3': c3, 'C4': c4, 'C5': c5}


class FPNNeck(nn.Module):
    """Feature Pyramid Network: C2..C5 -> P2..P5 with fixed out_channels."""

    def __init__(self, in_channels: Dict[str, int], out_channels: int = 256) -> None:
        super().__init__()
        self.out_channels = out_channels
        self.lat_c2 = nn.Conv2d(in_channels['C2'], out_channels, 1)
        self.lat_c3 = nn.Conv2d(in_channels['C3'], out_channels, 1)
        self.lat_c4 = nn.Conv2d(in_channels['C4'], out_channels, 1)
        self.lat_c5 = nn.Conv2d(in_channels['C5'], out_channels, 1)
        self.smooth = nn.ModuleDict({
            lvl: nn.Conv2d(out_channels, out_channels, 3, padding=1)
            for lvl in ('P2', 'P3', 'P4', 'P5')
        })

    def forward(self, feats: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        c2, c3, c4, c5 = feats['C2'], feats['C3'], feats['C4'], feats['C5']
        p5 = self.lat_c5(c5)
        p4 = self.lat_c4(c4) + F.interpolate(p5, size=c4.shape[-2:], mode='nearest')
        p3 = self.lat_c3(c3) + F.interpolate(p4, size=c3.shape[-2:], mode='nearest')
        p2 = self.lat_c2(c2) + F.interpolate(p3, size=c2.shape[-2:], mode='nearest')
        return {
            'P2': self.smooth['P2'](p2),
            'P3': self.smooth['P3'](p3),
            'P4': self.smooth['P4'](p4),
            'P5': self.smooth['P5'](p5),
        }


@dataclass
class DetectionHeadConfig:
    """Config for a lightweight detection head."""

    in_channels: int = 256
    num_anchors: int = 9
    num_classes: int = 1  # objectness demo
    hidden_channels: int = 256


class DetectionHead(nn.Module):
    """Two-branch detection head producing classification logits and box deltas."""

    def __init__(self, cfg: DetectionHeadConfig) -> None:
        super().__init__()
        k = cfg.num_anchors
        c = cfg.num_classes
        mid = cfg.hidden_channels

        self.cls_conv = nn.Sequential(
            nn.Conv2d(cfg.in_channels, mid, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.cls_head = nn.Conv2d(mid, k * c, kernel_size=1)

        self.reg_conv = nn.Sequential(
            nn.Conv2d(cfg.in_channels, mid, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.reg_head = nn.Conv2d(mid, k * 4, kernel_size=1)

    def forward(self, pyramid: Dict[str, torch.Tensor]) -> Dict[str, Dict[str, torch.Tensor]]:
        outputs: Dict[str, Dict[str, torch.Tensor]] = {}
        for lvl in ['P2', 'P3', 'P4', 'P5']:
            f = pyramid[lvl]
            cls_feat = self.cls_conv(f)
            reg_feat = self.reg_conv(f)
            outputs[lvl] = {
                'cls_logits': self.cls_head(cls_feat),
                'box_deltas': self.reg_head(reg_feat),
            }
        return outputs

# Complete forward pass: image -> ResNet50 -> FPN -> detection head
backbone = ResNet50Backbone().to(device)
fpn = FPNNeck(in_channels={'C2': 256, 'C3': 512, 'C4': 1024, 'C5': 2048}, out_channels=256).to(device)
head = DetectionHead(DetectionHeadConfig(in_channels=256, num_anchors=9, num_classes=1)).to(device)

img = torch.randn(1, 3, 224, 224, device=device)
pyr = fpn(backbone(img))
preds = head(pyr)

print('Head output shapes:')
for lvl in ['P2', 'P3', 'P4', 'P5']:
    cls_logits = preds[lvl]['cls_logits']
    box_deltas = preds[lvl]['box_deltas']
    print(' ', lvl, 'cls', tuple(cls_logits.shape), 'deltas', tuple(box_deltas.shape))


## 3.2 RoI Pooling

It was the original solution. It works, but it has a fundamental flaw: it forces floating-point region coordinates to align with the integer grid of the feature map via rounding — what the Mask R-CNN paper calls quantization. This introduces a spatial misalignment that is tolerable for bounding-box detection, but catastrophic for pixel-level tasks like instance segmentation where you need exact spatial correspondence.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_3_Key_Concepts_for_Deep_Learning_CV/roi-poll.png" width=800 heigth=400>

## 3.3 RoI Align (introduced in Mask R-CNN, 2017)
Eliminates all quantization. It treats the feature map as a continuous signal and samples it at precise floating-point positions using bilinear interpolation.

Given a region proposal in input-image coordinates and a stride factor S (e.g. S=16 for a VGG backbone), the algorithm is:

1. Map to feature-map space. Divide the proposal coordinates by the stride:
```text
x_feat = x_image / S       # e.g.  320 / 16 = 20.0
y_feat = y_image / S       # e.g.  192 / 16 = 12.0
w_feat = w_image / S       # e.g.   96 / 16 =  6.0
h_feat = h_image / S       # e.g.   64 / 16 =  4.0
```

2.  Divide into k × k bins. The output pool size is fixed at k×k (typically 7×7 or 14×14). Each bin covers exactly w_feat/k × h_feat/k of the feature map — again, floating-point dimensions.

3. Place sampling points inside each bin. For each bin, place a regular grid of n × n sampling points (usually n=2, so 4 points per bin). These points sit at fractional positions in the feature map.

4. Bilinear interpolation + max/avg pool. Each sampling point falls between four neighboring feature-map cells. Interpolate the value at that exact location using bilinear interpolation

The following example shows fixed size of 3x3

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_3_Key_Concepts_for_Deep_Learning_CV/roi-align2.png" width=600 heigth=400>


In [ ]:
# === Section 3 — Head + encode/decode + RoI Align ===

def _centers_from_xyxy(boxes: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    x1, y1, x2, y2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    w = (x2 - x1).clamp(min=1e-6)
    h = (y2 - y1).clamp(min=1e-6)
    cx = x1 + 0.5 * w
    cy = y1 + 0.5 * h
    return cx, cy, w, h


def encode_boxes(anchors: torch.Tensor, gt_boxes: torch.Tensor) -> torch.Tensor:
    """Encode gt_boxes relative to anchors as (dx, dy, dw, dh)."""
    a_cx, a_cy, a_w, a_h = _centers_from_xyxy(anchors)
    g_cx, g_cy, g_w, g_h = _centers_from_xyxy(gt_boxes)
    dx = (g_cx - a_cx) / a_w
    dy = (g_cy - a_cy) / a_h
    dw = torch.log(g_w / a_w)
    dh = torch.log(g_h / a_h)
    return torch.stack([dx, dy, dw, dh], dim=1)


def decode_boxes(anchors: torch.Tensor, deltas: torch.Tensor) -> torch.Tensor:
    """Decode (dx,dy,dw,dh) back into xyxy boxes."""
    a_cx, a_cy, a_w, a_h = _centers_from_xyxy(anchors)
    dx, dy, dw, dh = deltas[:, 0], deltas[:, 1], deltas[:, 2], deltas[:, 3]
    pred_cx = dx * a_w + a_cx
    pred_cy = dy * a_h + a_cy
    pred_w = torch.exp(dw) * a_w
    pred_h = torch.exp(dh) * a_h
    x1 = pred_cx - 0.5 * pred_w
    y1 = pred_cy - 0.5 * pred_h
    x2 = pred_cx + 0.5 * pred_w
    y2 = pred_cy + 0.5 * pred_h
    return torch.stack([x1, y1, x2, y2], dim=1)



# Quick encode/decode sanity check
anchors_demo = torch.tensor([[10, 10, 50, 50], [40, 20, 90, 70]], dtype=torch.float32)
gt_demo = torch.tensor([[12, 11, 52, 49], [42, 18, 100, 74]], dtype=torch.float32)
deltas = encode_boxes(anchors_demo, gt_demo)
decoded = decode_boxes(anchors_demo, deltas)
print('Decoded close to gt (allclose):', bool(torch.allclose(decoded, gt_demo, atol=1e-4)))

# RoI Pooling vs RoI Align visualization
C, H, W = 32, 20, 20
feat = torch.randn(1, C, H, W, device=device)
# roi tensor for torchvision.ops expects [batch_idx, x1, y1, x2, y2]
roi = torch.tensor([[0, 6.2, 7.3, 15.8, 17.1]], dtype=torch.float32, device=device)
out_size = (7, 7)
pooled = torchvision.ops.roi_pool(feat, roi, output_size=out_size, spatial_scale=1.0)
aligned = torchvision.ops.roi_align(feat, roi, output_size=out_size, spatial_scale=1.0, aligned=True)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(pooled[0, 0].detach().cpu().numpy(), cmap='magma')
axes[0].set_title('RoI Pool (channel 0)')
axes[0].axis('off')
axes[1].imshow(aligned[0, 0].detach().cpu().numpy(), cmap='magma')
axes[1].set_title('RoI Align (channel 0)')
axes[1].axis('off')
fig.suptitle('RoI Pooling vs RoI Align')
plt.tight_layout()
out_path = fig_dir / 'roi_pool_vs_align.png'
#plt.savefig(out_path)
plt.show()
#print('Saved:', out_path)



<a name='sec4'></a>

# Section 4 - NMS ( Non-Maximum Suppression )

<p>
An anchor-based (or otherwise dense) detector doesn't output a single bounding box per object — it outputs hundreds or thousands of candidate boxes, many of which heavily overlap around the same object. NMS is the post-processing algorithm that keeps only the best box per object and discards the rest. Without NMS, such a model would detect the same car 47 times. With NMS, it detects it once.

The Core Idea
Every candidate box comes with two pieces of information:

A bounding box [x1, y1, x2, y2]
A confidence score — how sure the model is that there's an object there

NMS uses the Intersection over Union (IoU) metric to decide which overlapping boxes refer to the same object.

If two boxes have a high IoU (e.g. > 0.5), they're almost certainly detecting the same object, *so only the higher-confidence one survives.*

</p>

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_3_Key_Concepts_for_Deep_Learning_CV/nms.png">

---

### Two things the picture above leaves out

**1. Real detectors run NMS *per class*, not class-agnostically.**
The version we implement below (and the one usually shown in tutorials) compares *every* box with every other box regardless of label. That will happily suppress a correct **person** detection because it overlaps a **bicycle** — which is exactly the situation you want to keep. Production code runs one independent NMS per class; in PyTorch that is `torchvision.ops.batched_nms(boxes, scores, idxs, iou_threshold)`, where `idxs` is the class index. It gets the same answer as a per-class loop by offsetting each class's coordinates into a disjoint region, so it stays a single fast call.

**Variants worth knowing:** **Soft-NMS** *decays* the score of an overlapping box (by `1 − IoU` or a Gaussian) instead of deleting it, which recovers detections in crowded scenes where two real objects genuinely overlap. **Class-agnostic** NMS is still the right choice for a class-agnostic proposal stage such as the RPN.

**2. NMS is no longer universal.**
It is a heuristic bolted on *after* the network, it is not differentiable, and its IoU threshold is one more hyperparameter. Detectors have been removing it:

| Detector | NMS? | Why |
|---|---|---|
| Faster R-CNN, SSD, RetinaNet, YOLOv5–v8 | required | dense one-to-many assignment produces duplicates |
| DETR (2020) and the whole DETR family | **none** | Hungarian one-to-one matching means each object is claimed by exactly one query |
| YOLOv10, YOLO26 | **none by default** | consistent dual assignment makes the model end-to-end |

So treat NMS as *"the classic way to deduplicate a dense detector"*, not as a mandatory stage of object detection.

## 4.1 NMS from scratch (synthetic demo)

In [ ]:
# NMS demo on synthetic boxes
def nms_pytorch(boxes: torch.Tensor, scores: torch.Tensor, iou_threshold: float = 0.5) -> torch.Tensor:
    """Class-agnostic NMS (xyxy).

    Class-agnostic = every box competes with every other box, whatever its label.
    That is what a class-agnostic proposal stage (an RPN) wants, but it is NOT what a
    multi-class detector wants: see the `batched_nms` comparison at the bottom of this cell.
    """
    if boxes.numel() == 0:
        return torch.empty((0,), dtype=torch.int64)

    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]
    areas = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    order = scores.argsort(descending=True)
    keep: List[int] = []

    while order.numel() > 0:
        i = int(order[0].item())
        keep.append(i)
        if order.numel() == 1:
            break

        rest = order[1:]
        xx1 = torch.maximum(x1[i], x1[rest])
        yy1 = torch.maximum(y1[i], y1[rest])
        xx2 = torch.minimum(x2[i], x2[rest])
        yy2 = torch.minimum(y2[i], y2[rest])

        inter = (xx2 - xx1).clamp(min=0) * (yy2 - yy1).clamp(min=0)
        iou = inter / (areas[i] + areas[rest] - inter + 1e-7)
        order = rest[iou <= iou_threshold]

    return torch.tensor(keep, dtype=torch.int64)


boxes_demo = torch.tensor([[10, 10, 50, 50], [12, 12, 48, 48], [60, 60, 100, 100], [55, 62, 98, 108]], dtype=torch.float32)
scores_demo = torch.tensor([0.9, 0.75, 0.8, 0.7], dtype=torch.float32)
keep_idx = nms_pytorch(boxes_demo, scores_demo, iou_threshold=0.5)
print('NMS kept indices:', keep_idx.tolist())
print('Matches torchvision.ops.nms:',
      sorted(keep_idx.tolist()) == sorted(torchvision.ops.nms(boxes_demo, scores_demo, 0.5).tolist()))

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.set_title('NMS from scratch (synthetic)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_xlim(0, 120)
ax.set_ylim(120, 0)
ax.grid(True, alpha=0.2)

for i, b in enumerate(boxes_demo):
    x1, y1, x2, y2 = b.tolist()
    color = 'lime' if i in keep_idx.tolist() else 'red'
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor=color, facecolor='none')
    ax.add_patch(rect)
    ax.text(x1 + 1, y1 + 12, f'{i}:{scores_demo[i]:.2f} {"" if i in keep_idx else "suppressed"}', color=color)

out_path2 = fig_dir / 'nms_synthetic_demo.png'
plt.tight_layout()
#plt.savefig(out_path2)
plt.show()
#print('Saved:', out_path2)

# ── Why class-agnostic NMS is the wrong default for a multi-class detector ────
# A person standing next to their bicycle: the two boxes genuinely overlap (IoU ~0.68),
# but they are two different objects and BOTH must survive.
CLASS_NAMES = {0: 'person', 1: 'bicycle'}
boxes_mc = torch.tensor([[30., 30., 90., 150.],    # person   (score 0.95)
                         [40., 60., 110., 160.]],  # bicycle  (score 0.88)
                        dtype=torch.float32)
scores_mc = torch.tensor([0.95, 0.88], dtype=torch.float32)
labels_mc = torch.tensor([0, 1], dtype=torch.int64)

print(f"\nIoU between the two boxes: {torchvision.ops.box_iou(boxes_mc, boxes_mc)[0, 1]:.3f}")

agnostic = torchvision.ops.nms(boxes_mc, scores_mc, iou_threshold=0.5)
per_class = torchvision.ops.batched_nms(boxes_mc, scores_mc, labels_mc, iou_threshold=0.5)

print('class-agnostic nms  kept:', [CLASS_NAMES[int(labels_mc[i])] for i in agnostic],
      '  <- the bicycle was wrongly suppressed')
print('per-class batched_nms kept:', [CLASS_NAMES[int(labels_mc[i])] for i in per_class],
      '  <- correct')

<a name='sec5'></a>

# Section 5 - Evaluation metrics in computer vision
---

Once a model is trained, **loss** tells you how optimisation is going — but **metrics** tell you whether predictions are *useful* on held-out data. The right metric depends on the **task** (classification vs detection) and on **what errors you care about** (missed objects, wrong labels, sloppy boxes, etc.).

Below are the most cited numbers in papers and leaderboards, with a consistent detection-friendly story that connects back to **IoU** from earlier sections.

---

## 5.1 Accuracy

**What it measures (classification).**  
For *single-label* image classification, **accuracy** is the fraction of images whose **predicted class** equals the **ground-truth class**:

$$\text{Accuracy} = \frac{\text{number of correct predictions}}{\text{total number of samples}} = \frac{\mathrm{TP} + \mathrm{TN}}{\mathrm{TP} + \mathrm{TN} + \mathrm{FP} + \mathrm{FN}}$$

- **Strength:** easy to communicate (“the model got 92% of images right”).
- **Limits:** with **imbalanced** classes, accuracy can look high while the model ignores rare classes. In that setting people also report **per-class accuracy**, **balanced accuracy**, or **macro-F1**.

> **Detection caveat:** Raw “classification accuracy” alone does not summarise object detection, because each image can have **many** objects and each prediction is a **(class, box)** pair. Detection benchmarks instead centre on **precision–recall curves** and **mAP** (below).

---


## 5.2 Recall

**What it measures.**  
Recall answers: *of all the ground-truth positives, how many did we find?*

$$\text{Recall} = \frac{\mathrm{TP}}{\mathrm{TP} + \mathrm{FN}}$$

- **TP (true positives):** predictions matched to a ground-truth object according to the task rules (for detection: usually **IoU** above a threshold **and** correct class).
- **FN (false negatives):** ground-truth objects with **no** acceptable matching prediction — the model **missed** them.

High recall means fewer **missed objects** — critical in applications like **autonomous driving** or **medical imaging**, where overlooking an instance can be catastrophic.

> **Pairing with precision.** In detection, **precision** = $\mathrm{TP} / (\mathrm{TP} + \mathrm{FP})$ asks how many predicted boxes are “real” vs spurious. Papers often report **precision–recall curves**; mAP summarises that curve in a single number.



---
## 5.3 Precision

**Precision** is a metric used to evaluate the quality of a classification model, especially in tasks like binary classification, object detection, and information retrieval.

> **Precision measures how many of the predicted positive instances are actually correct.**

$$\text{Precision} = \frac{\mathrm{TP}}{\mathrm{TP} + \mathrm{FP}}$$


Where:

* **TP (True Positives):** Correctly predicted positive cases
* **FP (False Positives):** Incorrectly predicted positive cases

Precision answers the question:

> *"When the model predicts positive, how often is it right?"*

A **high precision** means:

* Few false positives
* The model is **conservative** when predicting positives

### 📊 Example

Suppose a model predicts whether an email is spam:

|                     | Predicted Spam | Predicted Not Spam |
| ------------------- | -------------- | ------------------ |
| **Actual Spam**     | 40 (TP)        | 10 (FN)            |
| **Actual Not Spam** | 5 (FP)         | 45 (TN)            |

`P = ( 40/40+5 ) = 0.89`

👉 This means **89% of emails predicted as spam are actually spam**.

---

## 5.4 F1 Score

The **F1 Score** is a metric used in classification tasks to evaluate the balance between **Precision** and **Recall**. It is their **harmonic mean**.


<u>Formula</u>
---

$$
F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}
$$

<u>Why F1 Score Matters</u>
---

The F1 Score is useful when:

- The dataset is **imbalanced**
- Both **False Positives** and **False Negatives** matter
- You need a balance between precision and recall


<u>Interpretation</u>
---

| F1 Score | Meaning |
|---|---|
| 1.0 | Perfect model |
| 0.0 | Worst possible model |
| High | Good balance between precision and recall |
| Low | Poor classification performance |

### Key Takeaways

- F1 Score combines precision and recall
- Useful for imbalanced datasets
- Better than accuracy when class distribution is uneven
- Harmonic mean penalizes extreme values — a model with precision 1.0 and recall 0.0 scores 0, whereas the plain average would give it 0.5

---

## 5.5 mAP@50 (AP at IoU = 0.50)

**Context: object detection.**  
A predicted box is typically counted as a **true positive** for a ground-truth box if:

1. the **class label** matches, and  
2. $\mathrm{IoU}(\text{pred}, \text{gt}) \geq \tau$ for a fixed threshold $\tau$.

**mAP@50** uses **$\tau = 0.5$** — a relatively **lenient** localisation bar.

**How AP is built.**  
For **each class**, take the predictions of that class **from every image in the dataset**, pool them into one list, sort by **confidence**, and match them greedily to ground truths (each GT can be used once). Sweeping down that sorted list gives a **precision–recall curve**, and **Average Precision (AP)** summarises it.

> COCO does **not** take a raw trapezoidal area under that curve: `cocoeval.py` interpolates precision at **101 fixed recall levels** (0.00, 0.01, …, 1.00) and reports their **mean**.

**What "m" averages over.**

- **mAP averages AP over *classes*.** COCO-style mAP additionally averages over the **10 IoU thresholds** (§5.6).
- **AP is never computed per image and then averaged.** Detections from all images are pooled into a single PR curve *per class* — that is why a class with 3 objects in one image and 300 in another is scored as one population. Any pipeline that computes AP image-by-image and averages will produce numbers that match no published result.

| Aspect | mAP@50 |
|--------|--------|
| **Localisation strictness** | Moderate — box must overlap GT with **IoU ≥ 0.5** |
| **Averaged over** | Classes (one pooled PR curve per class) |
| **Typical use** | Quick sense of “does the detector find objects in roughly the right place?” |

---



## 5.6 mAP@[50:95] — “mAP50-95” or COCO-style mAP

**Stricter localisation.**  
Instead of a **single** IoU threshold, COCO-style **mAP** averages AP over **multiple** thresholds:

$$\tau \in \{0.50,\, 0.55,\, 0.60,\, \ldots,\, 0.95\}$$

(step **0.05**), then averages those APs. Informally:

$$\text{mAP} \approx \frac{1}{10}\sum_{k=0}^{9} \text{AP @ IoU} = (0.50 + 0.05k)$$

So **mAP50-95** rewards detectors that stay accurate when IoU must be **0.75**, **0.9**, etc. — not only at **0.5**. Two models can have **similar mAP@50** but very **different mAP50-95** if one produces **sloppier** boxes.

| Metric | What tightens / relaxes |
|--------|-------------------------|
| **mAP@50** | Easier localisation |
| **mAP@50–95** | Harder — must work across **many** IoU cut-offs |

---

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_3_Key_Concepts_for_Deep_Learning_CV/metrics.png">



## 5.7 One-page mental map

| Metric | Task focus | One-line intuition |
|--------|------------|--------------------|
| **Accuracy** | Classification | “Did we pick the **right label** for this image?” |
| **Precision** | Any task with positives | “When we **say** positive, how often are we right?” |
| **Recall** | Any task with positives | “Did we **find** most of the true objects (or positives)?” |
| **F1** | Classification (imbalanced) | “One number balancing precision and recall — the **harmonic mean**.” |
| **mAP@50** | Detection | “How good are we at **class + box** when IoU ≥ **0.5**?” |
| **mAP@50–95** | Detection | “How good are we when boxes must be **tighter**, averaged over many IoU levels?” |

---

### Takeaway

- Use **accuracy** for **global class correctness** in classification; complement it when data are **imbalanced**.  
- Use **precision** when a **false alarm** is costly, and **recall** when **missing instances** is costly; **F1** when both are.  
- Use **mAP@50** and **mAP@50–95** to report **object detection** quality with a clear, benchmark-aligned localisation story tied to **IoU** — always averaged over **classes**, never over images.

<a name='sec6'></a>

# Section 6 — Loss Functions (Multi-task Learning)
---


## 6.1 Classification Loss Functions

### 6.1.1 Cross-Entropy
---

Cross-entropy measures **how well a predicted probability distribution matches the true distribution**. It's the most common loss function for classification tasks.

### Intuition :

Imagine you're a forecaster. If you say "90% chance of rain" and it rains, great — low penalty. If you say "10% chance of rain" and it rains, you're punished heavily. Cross-entropy formalizes this: **confident wrong predictions are penalized much more than uncertain ones**.


### Formula :

For a single sample with true label $y$ and predicted probability $\hat{y}$:

**Binary classification:**
$$\mathcal{L} = -\left[ y \log(\hat{y}) + (1 - y) \log(1 - \hat{y}) \right]$$

**Multi-class classification** (C classes):
$$\mathcal{L} = -\sum_{c=1}^{C} y_c \log(\hat{y}_c)$$

Since $y_c$ is one-hot (only one class is 1), this simplifies to just:
$$\mathcal{L} = -\log(\hat{y}_{\text{true class}})$$


### Why the `-log`?

The key is the shape of $-\log(p)$:

| Predicted prob for true class | Loss |
|---|---|
| 0.99 | ≈ 0.01 (almost no penalty) |
| 0.50 | ≈ 0.69 |
| 0.10 | ≈ 2.30 |
| 0.01 | ≈ 4.60 (heavy penalty) |

It's **asymmetric and nonlinear** — being confidently wrong is catastrophic.

### 6.1.2 Focal Loss — Fighting class imbalance
---

### The problem it solves

<p>
In a single-stage detector (like RetinaNet), the model evaluates ~100,000 candidate locations per image.

The overwhelming majority are easy negatives — patches of sky, road, background. Only a handful contain actual objects.

Standard Cross-Entropy gives all examples equal weight. The result: the model drowns in easy background examples. The gradient signal from the few hard, interesting examples gets buried.

</p>

---

### The Mechanism of Focusing

The core innovation of Focal Loss lies in its ability to dynamically scale the penalty assigned to each sample based on the model's confidence.

In a typical supervised learning scenario, a detector might evaluate thousands of candidate locations in an image. Since most of these locations contain no objects, a standard loss function accumulates many small error signals from these easy negatives, which can drown out the valuable signal from the few positive instances.

Focal Loss introduces a modulating factor that decays the loss contribution as the confidence in the correct class increases.

This means that if a model is already 99% sure that a background patch is indeed background, the loss for that patch is reduced to near zero.

Consequently, the model weights are updated primarily based on misclassified samples or those where the model is uncertain.

```text
FL(p_t) = -α_t · (1 - p_t)^γ · log(p_t)
           ───┬───   ───┬────
              │         └── focusing term: suppresses easy examples
              └── class balancing weight (α ≈ 0.25 for foreground)
```

The ```(1 - p_t)^γ``` term is the key insight. When the model is confident *(p_t → 1)*, this *factor → 0*, effectively muting the loss contribution from easy examples. Hard examples (where p_t is low) retain nearly their full loss signal.

---

### Who actually uses it

Focal Loss is **RetinaNet's** contribution (Lin et al., *Focal Loss for Dense Object Detection*, 2017) — it is what let a one-stage detector match two-stage accuracy without a sampling stage. It is also **DETR's** classification loss.

> ⚠️ **The YOLO line took a different route.** Ultralytics' YOLOv5/v8/v11 classify with plain
> `BCEWithLogitsLoss` and handle imbalance through **label assignment** instead — YOLOv5's own
> `hyp.scratch-low.yaml` ships `fl_gamma: 0.0`, i.e. focal loss **disabled by default**, and
> v8 onwards uses a task-aligned assigner. **YOLO26 goes further and removes Distribution Focal
> Loss (DFL) from the head altogether.** So "YOLO uses focal loss" is a common and incorrect
> shortcut: the imbalance problem is real for all dense detectors, but focal loss is only one of
> the answers to it.


### 6.1.3 Focal Loss vs. Cross-Entropy Loss

<p>
Cross-Entropy is the foundational metric for classification that penalizes predictions based on logarithmic error.

Focal Loss is strictly an extension of Cross-Entropy; if the focusing parameter γ is set to zero, it mathematically reverts to (α-weighted) standard Cross-Entropy.

The key distinction is Focal Loss's ability to automatically down-weight easy negatives, which is what made RetinaNet's dense one-stage design work on an imbalanced dataset like COCO.
</p>

> Remember from 6.1.2 that "focal loss ⇒ every modern detector" does not follow. RetinaNet and
> the DETR family use it; the Ultralytics YOLO line does not, and YOLO26 removes DFL from its
> head as well.



## 6.2 Loss Functions for Bounding Box Detection
---

A detector predicts 4 numbers per box. The loss function measures how wrong those 4 numbers are — and its shape determines what the model actually learns to optimize. The field has evolved through several generations, each fixing a blind spot in the previous one.


### What is the model actually predicting?

**Anchor-based** detectors predict **encoded offsets** relative to an anchor box, which are then decoded into absolute coordinates (this is the encoding from §2.2.4):

```text
Predicted:   (tx, ty, tw, th)   ← raw network outputs, anchor-normalised
Decoded:     (cx, cy, w, h)     ← actual box coordinates
             or (x1, y1, x2, y2)
```

**Anchor-free** detectors (§2.2.5) instead predict `(l, t, r, b)` distances from a feature-map location to the four edges, and DETR-style models predict normalised `(cx, cy, w, h)` in `[0,1]` directly from a query.

Whichever parametrisation is used, the regression loss is computed either **in the encoded space** (Smooth L1 / L1) or **on the decoded boxes** (the IoU family). That distinction matters for the next two subsections.

---


### 6.2.1 L2 Loss (MSE)

The naive starting point: minimize the sum of squared coordinate errors.

```text
L_L2 = (x1_pred − x1_gt)² + (y1_pred − y1_gt)²
      + (x2_pred − x2_gt)² + (y2_pred − y2_gt)²
```

It works, but has two fatal problems:

- A single outlier box with a large error generates a **massive gradient** (`error²` blows up), destabilizing training.
- It treats all 4 coordinates as independent — there's **no notion of geometric overlap**. A box 10px off in `x` gets the same penalty whether that shifts it completely off the object or barely changes the overlap.

---



### 6.2.2 Smooth L1 (Huber Loss)

Introduced in Faster R-CNN (2015). Fixes the outlier explosion by stitching L2 near zero with L1 for large errors:

```text
           ⎧  0.5 · x²               if |x| < δ    ← smooth, stable gradient
SL1(x) =  ⎨
           ⎩  δ · (|x| − 0.5·δ)     if |x| ≥ δ    ← linear, bounded gradient
```

Applied per-coordinate, summed — **on the encoded deltas of §2.2.4, not on pixels**:

```text
L_SL1 = SL1(Δtx) + SL1(Δty) + SL1(Δtw) + SL1(Δth)

where   tx = (gx − ax)/aw ,  ty = (gy − ay)/ah ,  tw = log(gw/aw) ,  th = log(gh/ah)
```

The key property: the gradient is capped at `±δ` for large errors. A wildly wrong box prediction can't explode the gradient and corrupt the batch.

**Note that this encoding already normalises by the anchor size.** A 10 px centre error on a 20 px anchor gives `tx = 0.5`; the same 10 px error on a 500 px anchor gives `tx = 0.02` — a 25× smaller penalty. So Smooth L1 is *not* blind to object scale (see 6.2.8 for the version of this claim that *is* true).

Its real limitation is different: it still treats the 4 numbers as **independent scalars** with no notion of overlap. The quantity we evaluate on is IoU, and Smooth L1 optimises something else. That decoupling from the evaluation metric is exactly what the IoU-family losses fix.

---

### 6.2.3 IoU Loss

The natural idea: just minimize `1 − IoU` directly.

```text
L_IoU = 1 − IoU(B_pred, B_gt)

         area(B_pred ∩ B_gt)
IoU =  ─────────────────────────
         area(B_pred ∪ B_gt)
```

This is **scale-invariant** — a small box 5px off is penalized similarly to a large box 50px off, which is exactly right. And it directly optimizes the metric you actually care about (IoU at test time).

The critical failure: **when boxes don't overlap, IoU = 0, gradient = 0.** The loss is flat everywhere outside the ground truth box. The model gets no signal about which direction to move a non-overlapping prediction.

```text
Box A = [0,0,10,10]    Box B = [20,0,30,10]  →  IoU = 0,  Loss = 1
Box A = [0,0,10,10]    Box B = [200,0,210,10] → IoU = 0,  Loss = 1  ← identical!
```

Two completely different situations, same loss, same gradient (zero). Training stalls whenever boxes don't overlap at initialization.
<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_3_Key_Concepts_for_Deep_Learning_CV/iou.webp">


### 6.2.4 GIoU Loss
---

*Generalized IoU* (Rezatofighi et al., CVPR 2019). Adds a penalty based on the **smallest enclosing box** `C` that contains both boxes:

```text
              |C| − |B_pred ∪ B_gt|
GIoU = IoU − ─────────────────────
                      |C|

L_GIoU = 1 − GIoU    ∈ [0, 2]
```

The penalty term `(|C| − |A ∪ B|) / |C|` is the fraction of the enclosing box that's empty — not covered by either prediction or target. When boxes are far apart, this term grows large, giving a meaningful gradient even with zero overlap.

| Situation | IoU | GIoU | Gradient? |
|-----------|-----|------|-----------|
| Perfect overlap | 1 | 1 | zero loss |
| Partial overlap | 0.4 | 0.3 | yes — from both terms |
| No overlap, nearby | 0 | −0.1 | yes — from enclosing box |
| No overlap, far | 0 | −0.8 | yes — stronger signal |

GIoU ∈ `[−1, 1]` always. The model always knows which way to move.

**Remaining weakness**: when one box is fully inside the other (nested boxes), GIoU collapses to plain IoU — the enclosing box penalty vanishes and the gradient becomes weak.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_3_Key_Concepts_for_Deep_Learning_CV/giou.webp">


### 6.2.5 DIoU Loss
---

*Distance IoU* (Zheng et al., AAAI 2020). Adds a penalty for the **distance between box centers**:
<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_3_Key_Concepts_for_Deep_Learning_CV/diou.webp">
```text
                    ρ²(b_pred, b_gt)
DIoU = IoU −  ─────────────────────────
                       c²

where  ρ = Euclidean distance between centers
       c = diagonal of enclosing box C

L_DIoU = 1 − DIoU
```

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_3_Key_Concepts_for_Deep_Learning_CV/diou2.webp">

The center-distance term pushes the predicted box's center toward the target's center, even when one box is nested inside the other. Convergence is faster than GIoU because the gradient signal is more direct.

---


### 6.2.6 CIoU Loss

*Complete IoU* (same paper as DIoU). Adds a third term: **aspect ratio consistency**.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_3_Key_Concepts_for_Deep_Learning_CV/ciou.webp" width=400 heigth=400>

CIoU simultaneously penalizes three geometric discrepancies:

```text
Overlap area  +  Center distance  +  Aspect ratio
   (IoU)           (ρ²/c²)              (α·v)
```

**The full formula** — the two terms the summary above hides are `v` and `α`:

$$\mathcal{L}_{\mathrm{CIoU}} = 1 - \mathrm{IoU} + \frac{\rho^2(b,\,b^{gt})}{c^2} + \alpha\, v$$

$$v = \frac{4}{\pi^2}\left(\arctan\frac{w^{gt}}{h^{gt}} - \arctan\frac{w}{h}\right)^{2}
\qquad
\alpha = \frac{v}{(1 - \mathrm{IoU}) + v}$$

- **`v` measures the aspect-ratio mismatch.** `arctan(w/h)` is the angle of the box's diagonal, so `v` compares *shapes* while ignoring *size*. The `4/π²` factor normalises it: the two arctangents each live in `(0, π/2)`, so their difference is bounded by `π/2` and `v ∈ [0, 1)`.
- **`α` is a trade-off weight, not a hyperparameter** — it is computed from the current IoU. When the boxes barely overlap, `IoU → 0`, so `(1 − IoU) → 1` and `α = v/(1+v) ≈ 0` for small `v`: the aspect-ratio term is switched **off**. As IoU grows, `(1 − IoU) → 0` and `α → 1`: the aspect-ratio term is switched **on**.
- **The effect is a curriculum baked into the loss:** fix the overlap and the centre first, and only worry about matching the shape once the boxes are actually on top of each other. In the reference implementation `α` is treated as a constant during backprop (no gradient flows through it), so it acts purely as a gate.

**Where it is used:** CIoU is the box-regression loss of **YOLOv5, YOLOv8 and YOLO11**. The DETR family (including **DINO-DETR**) does *not* use it — those models regress boxes with **L1 + GIoU**.




### 6.2.7 Full comparison at a glance
---

| Loss | IoU-scale-invariant | No-overlap gradient | Center-aware | Aspect-ratio aware | Used in |
|------|:-:|:-:|:-:|:-:|---------|
| L2 (MSE) | no | yes | no | no | early detectors |
| Smooth L1 | no | yes | no | no | Faster R-CNN, SSD, older YOLO |
| L1 | no | yes | no | no | DETR (with GIoU) |
| IoU | yes | **no** | no | no | baseline |
| GIoU | yes | yes | no | no | DETR, Deformable DETR, DINO-DETR (with L1) |
| DIoU | yes | yes | yes | no | YOLOv4 |
| CIoU | yes | yes | yes | yes | YOLOv5, YOLOv8, YOLO11 |

### 6.2.8 What "scale-invariant" actually means here

This is a claim that is easy to get wrong, so be precise about it.

**Smooth L1 is *not* computed on raw pixel differences.** As §6.2.2 showed, Faster R-CNN applies it to the *encoded* deltas, which are already divided by the anchor's width and height (and log-scaled for `w`/`h`). A 10 px centre error on a 20 px anchor produces `tx = 0.5`; on a 500 px anchor, `tx = 0.02`. Those are **not** penalised equally — the anchor encoding is itself a scale normalisation.

**What Smooth L1 lacks is scale-invariance *in the IoU sense*.** It optimises four independent scalars and has **no notion of overlap at all**. Two consequences:

- The loss value tells you nothing about the metric you will be scored on. A pair of predictions with identical `L_SL1` can have very different IoU, depending on *which* coordinates were wrong and in which direction.
- The normalisation depends on the **anchor**, not on the objects. An anchor-free model, or an anchor badly matched to the object, changes the effective scaling.

**IoU-family losses have neither problem** — they measure a geometric overlap *ratio*, so a 50% overlap is a loss of 0.5 whatever the absolute size of the object, and the quantity being minimised is (a monotone function of) the quantity being evaluated.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_3_Key_Concepts_for_Deep_Learning_CV/loss_histogram.webp" width=400 heigh=400>


---

### 6.2.9 How they combine in a real detector

Modern detectors use a weighted combination — classification and box regression have separate loss branches:

```text
Total Loss = λ_box · L_box  +  λ_cls · L_cls  +  λ_obj · L_obj
```

**YOLOv5** (`data/hyps/hyp.scratch-low.yaml`, the default recipe):

```text
  L_box  = CIoU loss                     λ_box = 0.05
  L_cls  = BCEWithLogitsLoss             λ_cls = 0.5      (fl_gamma: 0.0 → focal loss OFF)
  L_obj  = BCE on the objectness score   λ_obj = 1.0
```

Two things to read off that table, because both contradict the "obvious" story:

1. **Classification is plain BCE, not focal loss.** `fl_gamma: 0.0` disables the focusing term by default (§6.1.2).
2. **The box term has the *smallest* gain, not the largest.** `λ_box = 0.05` against `λ_obj = 1.0`.

> **The gains are not statements about importance.** They are **numeric-range compensation**. The
> three terms are summed over different populations and live on different scales: `L_box` is a CIoU
> loss in `[0,2]` summed over the (few) positive anchors, while `L_obj` is a BCE averaged over
> *every* anchor at every scale. The coefficients equalise the size of the three gradient
> contributions. Read them as calibration constants, and expect them to change with the number of
> classes, the number of scales and the batch size — which is exactly why they are in a
> hyperparameter file.

**Where the "box weighted 5×" figure comes from:** it is **DETR's**, not YOLO's. DETR's
set-prediction loss uses `λ_cls = 1`, `λ_L1 = 5`, `λ_giou = 2` — and there the box term genuinely is
up-weighted, because DETR regresses normalised coordinates in `[0,1]` whose raw L1 magnitudes are
tiny. Different parametrisation, different calibration.

---

### The intuition in one sentence per loss

- **Smooth L1** — *"be roughly right on all 4 coordinates, don't let outliers blow up training"*
- **IoU Loss** — *"optimize the overlap directly, but you're stuck if boxes don't touch"*
- **GIoU Loss** — *"always move toward the target, even from across the image"*
- **DIoU Loss** — *"get your center right first, then worry about size"*
- **CIoU Loss** — *"match center, shape, AND aspect ratio — be completely right"*

### 6.2.10 Hands-on: watching IoU loss die

The claim in §6.2.3 — *IoU loss has zero gradient when the boxes are disjoint* — is the reason GIoU and DIoU exist, so let's verify it rather than take it on faith.

We evaluate all four IoU-family losses on **three** prediction/ground-truth pairs, all with identically-shaped boxes so the CIoU aspect-ratio term stays out of the way:

| Case | Prediction vs GT |
|---|---|
| `overlapping` | boxes partially overlap |
| `disjoint-nearby` | no overlap, ~1 box-width apart |
| `disjoint-far` | no overlap, ~6 box-widths apart |

What to look for:

- `L_IoU` should read **exactly 1.0000** for *both* disjoint cases — it cannot tell "just missed" from "the other side of the image".
- `L_GIoU` and `L_DIoU` should keep **growing** with the distance.
- The gradient check at the end should print `|grad| = 0.000000` for IoU loss on the disjoint pairs, and a non-zero value for GIoU. That zero is a stalled training run.

In [ ]:
import torch
import torch.nn.functional as F
from torchvision.ops import (
    box_iou,
    generalized_box_iou,
    distance_box_iou,
    complete_box_iou,
)

# ── Smooth L1 — computed on the ENCODED deltas, not on pixels (see 6.2.2) ─────
pred_offsets = torch.tensor([[0.1, -0.2, 0.05, 0.3]])   # (tx, ty, tw, th)
gt_offsets   = torch.tensor([[0.0,  0.0, 0.00, 0.0]])
smooth_l1 = F.smooth_l1_loss(pred_offsets, gt_offsets, beta=1.0)
print(f"Smooth L1 on encoded deltas : {smooth_l1:.4f}\n")


# ── IoU family — computed on the DECODED box coordinates (x1,y1,x2,y2) ────────
def iou_family_losses(pred: torch.Tensor, gt: torch.Tensor):
    """Return (L_IoU, L_GIoU, L_DIoU, L_CIoU) for matched pred/gt boxes."""
    return (
        (1 - box_iou(pred, gt).diag()).mean(),               # plain IoU loss
        (1 - generalized_box_iou(pred, gt).diag()).mean(),   # GIoU
        (1 - distance_box_iou(pred, gt).diag()).mean(),      # DIoU
        (1 - complete_box_iou(pred, gt).diag()).mean(),      # CIoU
    )


# Ground truth: 140 x 110 box. Every prediction below is also 140 x 110, so the
# CIoU aspect-ratio term v is 0 and only overlap / centre distance move.
gt_boxes = torch.tensor([[80., 90., 220., 200.]])

cases = {
    "overlapping":     torch.tensor([[ 50.,  60., 190., 170.]]),   # partial overlap
    "disjoint-nearby": torch.tensor([[240.,  90., 380., 200.]]),   # no overlap, ~1 width away
    "disjoint-far":    torch.tensor([[900.,  90., 1040., 200.]]),  # no overlap, ~6 widths away
}

print(f"{'case':<17}{'IoU':>8}{'L_IoU':>9}{'L_GIoU':>9}{'L_DIoU':>9}{'L_CIoU':>9}")
print("-" * 61)
for name, pred in cases.items():
    iou = box_iou(pred, gt_boxes).diag().item()
    l_iou, l_giou, l_diou, l_ciou = iou_family_losses(pred, gt_boxes)
    print(f"{name:<17}{iou:>8.3f}{l_iou:>9.4f}{l_giou:>9.4f}{l_diou:>9.4f}{l_ciou:>9.4f}")

print(
    "\nL_IoU is pinned at exactly 1.0000 for both disjoint cases: the loss cannot\n"
    "distinguish 'just missed' from 'other side of the image'. GIoU and DIoU keep\n"
    "growing, because their extra term measures the enclosing box / centre distance."
)


# ── The consequence: is there a gradient to descend? ──────────────────────────
def grad_norm(loss_fn, pred: torch.Tensor, gt: torch.Tensor) -> float:
    """|dL/d(pred box)| summed over the 4 coordinates."""
    p = pred.clone().requires_grad_(True)
    (1 - loss_fn(p, gt).diag()).mean().backward()
    return p.grad.abs().sum().item()


print("\nGradient w.r.t. the predicted coordinates:")
print(f"{'case':<17}{'|grad| IoU':>13}{'|grad| GIoU':>14}{'|grad| DIoU':>14}")
print("-" * 58)
for name, pred in cases.items():
    print(
        f"{name:<17}"
        f"{grad_norm(box_iou, pred, gt_boxes):>13.6f}"
        f"{grad_norm(generalized_box_iou, pred, gt_boxes):>14.6f}"
        f"{grad_norm(distance_box_iou, pred, gt_boxes):>14.6f}"
    )

print(
    "\nZero gradient on the disjoint rows = no learning signal at all. A detector\n"
    "initialised with boxes that miss their targets would never recover with plain\n"
    "IoU loss; GIoU/DIoU still point it in the right direction."
)

<a name='references'></a>

# References
---

- Shaoqing Ren 2015 | [Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks](https://arxiv.org/pdf/1506.01497) — anchors, the RPN assignment rules and Smooth L1 on encoded deltas
- Kaiming He 2017 | [Mask R-CNN](https://arxiv.org/pdf/1703.06870) — RoI Align
- Tsung-Yi Lin 2018 | [Focal Loss for Dense Object Detection](https://arxiv.org/pdf/1708.02002) — RetinaNet
- Hamid Rezatofighi 2019 | [Generalized Intersection over Union: A Metric and A Loss for Bounding Box](https://arxiv.org/pdf/1902.09630)
- Zhaohui Zheng 2019 | [Distance-IoU Loss: Faster and Better Learning for Bounding Box Regression](https://arxiv.org/pdf/1911.08287) — DIoU **and** CIoU
- Zhi Tian 2019 | [FCOS: Fully Convolutional One-Stage Object Detection](https://arxiv.org/pdf/1904.01355) — anchor-free l/t/r/b regression
- Nicolas Carion 2020 | [End-to-End Object Detection with Transformers](https://arxiv.org/pdf/2005.12872) — DETR, Hungarian matching, NMS-free detection
- [Learning non-maximum suppression](https://arxiv.org/pdf/1705.02950)
- [COCO evaluation code (`cocoeval.py`)](https://github.com/cocodataset/cocoapi/blob/master/PythonAPI/pycocotools/cocoeval.py) — the 101-point AP definition
- [YOLO26 model documentation](https://docs.ultralytics.com/models/yolo26/) — end-to-end NMS-free, DFL removed
- [ROI Pooling](https://deepsense.ai/blog/region-of-interest-pooling-explained/)